# Model Comparison
#### LightGBM vs Random Forest vs XGBoost vs CatBoost

In [10]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import scipy.stats

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.ensemble import RandomForestClassifier

from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [2]:
# ========================
# 데이터 지정 (Cleaning)
# ========================
DATA_DIR = Path('../Data')
train_df = pd.read_csv(DATA_DIR / "train_cleaning.csv")
test_df = pd.read_csv(DATA_DIR / "test_cleaning.csv")

target_col = "fraud"   # 필요하면 "TARGET" 등으로 수정

# Train / Test 분리
X = train_df.drop(columns=[target_col]).copy()
y = train_df[target_col].copy()

X_test = test_df.copy()   # test에는 target 없음

print("Train X shape:", X.shape)
print("Train y shape:", y.shape)
print("Test X shape :", X_test.shape)
print("Positive ratio:", y.mean().round(4))

Train X shape: (18000, 125)
Train y shape: (18000,)
Test X shape : (12000, 125)
Positive ratio: 0.1582


In [3]:
# 예측 확률을 기반으로 threshold를 적용해 
# 이진 분류 성능 지표(ROC-AUC, PR-AUC, F1, Precision, Recall)를 계산
def evaluate_binary_classification(y_true, y_prob, threshold=0.4):
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0)
    }

In [4]:
# 모델(LGB, RF, XGB, CatBoost)을 학습·평가하고 OOF 성능 및 예측값을 반환
def run_cv_models(X, y, n_splits=5, random_state=42, threshold=0.4):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    model_dict = {
        "LightGBM": "lgb",
        "RandomForest": "rf",
        "XGBoost": "xgb",
        "CatBoost": "cat"
    }

    all_results = []
    oof_predictions = {}
    fitted_models = {name: [] for name in model_dict.keys()}

    for model_name, model_type in model_dict.items():
        print(f"\n{'='*25} {model_name} {'='*25}")

        oof_pred = np.zeros(len(X))
        fold_metrics = []

        for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
            print(f"\n{'='*20} Fold {fold} {'='*20}")

            X_train = X.iloc[train_idx].copy()
            X_valid = X.iloc[valid_idx].copy()
            y_train = y.iloc[train_idx].copy()
            y_valid = y.iloc[valid_idx].copy()

            neg = (y_train == 0).sum()
            pos = (y_train == 1).sum()
            scale_pos_weight = (neg / pos) * 1.1 if pos > 0 else 1.0

            # =====================================
            # 1) LightGBM: modeling_v4와 동일 조건
            # =====================================
            if model_type == "lgb":
                model = LGBMClassifier(
                    objective="binary",
                    metric="auc",
                    boosting_type="gbdt",
                    learning_rate=0.03,
                    num_leaves=64,
                    max_depth=-1,
                    min_child_samples=20,
                    subsample=0.8,
                    subsample_freq=1,
                    colsample_bytree=0.8,
                    reg_alpha=0.1,
                    reg_lambda=0.5,
                    n_estimators=10000,
                    random_state=random_state,
                    n_jobs=-1,
                    scale_pos_weight=scale_pos_weight,
                    verbosity=-1,
                )

                model.fit(
                    X_train,
                    y_train,
                    eval_set=[(X_valid, y_valid)],
                    eval_metric="auc",
                    callbacks=[
                        early_stopping(stopping_rounds=300, verbose=False),
                        log_evaluation(period=0)
                    ]
                )

                valid_prob = model.predict_proba(X_valid)[:, 1]

            # ========================
            # 2) RandomForest: 비교용
            # ========================
            elif model_type == "rf":
                model = RandomForestClassifier(
                    n_estimators=1000,
                    max_depth=12,
                    min_samples_split=20,
                    min_samples_leaf=10,
                    max_features="sqrt",
                    class_weight="balanced_subsample",
                    bootstrap=True,
                    random_state=random_state,
                    n_jobs=-1
                )

                model.fit(X_train, y_train)
                valid_prob = model.predict_proba(X_valid)[:, 1]

            # ====================
            # 3) XGBoost: 비교용
            # ====================
            elif model_type == "xgb":
                model = XGBClassifier(
                    objective="binary:logistic",
                    eval_metric="auc",
                    learning_rate=0.03,
                    n_estimators=10000,
                    max_depth=6,
                    min_child_weight=3,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    reg_alpha=0.1,
                    reg_lambda=0.5,
                    scale_pos_weight=scale_pos_weight,
                    random_state=random_state,
                    n_jobs=-1,
                    tree_method="hist"
                )

                model.fit(
                    X_train,
                    y_train,
                    eval_set=[(X_valid, y_valid)],
                    verbose=False
                )

                valid_prob = model.predict_proba(X_valid)[:, 1]

            # ====================
            # 4) CatBoost: 비교용
            # ====================
            elif model_type == "cat":
                model = CatBoostClassifier(
                    loss_function="Logloss",
                    eval_metric="AUC",
                    learning_rate=0.03,
                    iterations=10000,
                    depth=6,
                    l2_leaf_reg=0.5,
                    bootstrap_type="Bernoulli",
                    subsample=0.8,
                    scale_pos_weight=scale_pos_weight,
                    random_seed=random_state,
                    verbose=0
                )

                model.fit(
                    X_train,
                    y_train,
                    eval_set=(X_valid, y_valid),
                    verbose=False
                )

                valid_prob = model.predict_proba(X_valid)[:, 1]

            oof_pred[valid_idx] = valid_prob
            fitted_models[model_name].append(model)

            fold_result = evaluate_binary_classification(
                y_valid, valid_prob, threshold=threshold
            )
            fold_result["fold"] = fold
            fold_metrics.append(fold_result)

            print(
                f"Fold {fold} | "
                f"ROC-AUC: {fold_result['roc_auc']:.4f} | "
                f"PR-AUC: {fold_result['pr_auc']:.4f} | "
                f"F1: {fold_result['f1']:.4f} | "
                f"Precision: {fold_result['precision']:.4f} | "
                f"Recall: {fold_result['recall']:.4f}"
            )

            if model_type == "lgb" and hasattr(model, "best_iteration_"):
                print(f"Best Iteration: {model.best_iteration_}")

        # OOF 평가
        oof_result = evaluate_binary_classification(y, oof_pred, threshold=threshold)

        fold_df = pd.DataFrame(fold_metrics)
        summary = {
            "model": model_name,
            "oof_roc_auc": oof_result["roc_auc"],
            "oof_pr_auc": oof_result["pr_auc"],
            "oof_f1": oof_result["f1"],
            "oof_precision": oof_result["precision"],
            "oof_recall": oof_result["recall"],
            "cv_roc_auc_mean": fold_df["roc_auc"].mean(),
            "cv_roc_auc_std": fold_df["roc_auc"].std(),
            "cv_pr_auc_mean": fold_df["pr_auc"].mean(),
            "cv_pr_auc_std": fold_df["pr_auc"].std(),
            "cv_f1_mean": fold_df["f1"].mean(),
            "cv_f1_std": fold_df["f1"].std(),
        }

        all_results.append(summary)
        oof_predictions[model_name] = oof_pred

        print(f"\n[{model_name}] OOF Results")
        print(f"ROC-AUC   : {oof_result['roc_auc']:.4f}")
        print(f"PR-AUC    : {oof_result['pr_auc']:.4f}")
        print(f"F1        : {oof_result['f1']:.4f}")
        print(f"Precision : {oof_result['precision']:.4f}")
        print(f"Recall    : {oof_result['recall']:.4f}")

    results_df = pd.DataFrame(all_results).sort_values(
        by=["oof_pr_auc", "oof_roc_auc"], ascending=False
    ).reset_index(drop=True)

    return results_df, oof_predictions, fitted_models

In [5]:
# 모델 교차검증 실행 + 성능 비교 결과 출력
RANDOM_STATE = 2542

results_df, oof_preds, fitted_models = run_cv_models(
    X=X,
    y=y,
    n_splits=5,
    random_state=RANDOM_STATE,
    threshold=0.4
)

print(results_df.round(4))


========================= LightGBM =========================

==================== Fold 1 ====================
Fold 1 | ROC-AUC: 0.6808 | PR-AUC: 0.2800 | F1: 0.3469 | Precision: 0.2281 | Recall: 0.7241
Best Iteration: 100

==================== Fold 2 ====================
Fold 2 | ROC-AUC: 0.7111 | PR-AUC: 0.3152 | F1: 0.3610 | Precision: 0.2396 | Recall: 0.7311
Best Iteration: 83

==================== Fold 3 ====================
Fold 3 | ROC-AUC: 0.6890 | PR-AUC: 0.2730 | F1: 0.3529 | Precision: 0.2375 | Recall: 0.6860
Best Iteration: 135

==================== Fold 4 ====================
Fold 4 | ROC-AUC: 0.6865 | PR-AUC: 0.2854 | F1: 0.3451 | Precision: 0.2343 | Recall: 0.6544
Best Iteration: 190

==================== Fold 5 ====================
Fold 5 | ROC-AUC: 0.7042 | PR-AUC: 0.2860 | F1: 0.3673 | Precision: 0.2508 | Recall: 0.6860
Best Iteration: 189

[LightGBM] OOF Results
ROC-AUC   : 0.6934
PR-AUC    : 0.2830
F1        : 0.3545
Precision : 0.2378
Recall    : 0.6963

=========

In [6]:
# Recall@TopK 함수
def recall_at_k(y_true, y_prob, top_ratio=0.10):
    n_top = int(len(y_prob) * top_ratio)
    idx = np.argsort(y_prob)[::-1][:n_top]

    y_true_top = np.array(y_true)[idx]
    total_positive = np.sum(y_true)

    if total_positive == 0:
        return 0.0

    return np.sum(y_true_top) / total_positive

In [7]:
# 모델별 Recall@TopK
topk_rows = []

for model_name, pred in oof_preds.items():
    row = {
        "model": model_name,
        "Recall@Top5%": recall_at_k(y, pred, top_ratio=0.05),
        "Recall@Top10%": recall_at_k(y, pred, top_ratio=0.10),
        "Recall@Top20%": recall_at_k(y, pred, top_ratio=0.20),
    }
    topk_rows.append(row)

topk_df = pd.DataFrame(topk_rows).sort_values("Recall@Top10%", ascending=False).reset_index(drop=True)
print(topk_df.round(4))

          model  Recall@Top5%  Recall@Top10%  Recall@Top20%
0      CatBoost        0.1296         0.2261         0.3887
1      LightGBM        0.1211         0.2219         0.3789
2  RandomForest        0.1190         0.2145         0.3662
3       XGBoost        0.0945         0.1847         0.3269


## DeLong Test
OOF 예측값 기준으로 두 모델의 ROC-AUC 차이가 통계적으로 유의미한지 검정

In [15]:
# --- github.com/yandexdataschool/roc_comparison GitHub 코드 이용 ---

# 동점(tie)을 고려한 midrank 계산 함수
def compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)

    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1)
        i = j

    T2 = np.empty(N, dtype=float)
    T2[J] = T + 1
    return T2

In [12]:
# Fast DeLong 구현: AUC와 covariance 계산
def fast_delong(predictions_sorted_transposed, label_1_count):
    m = label_1_count
    n = predictions_sorted_transposed.shape[1] - m
    positive_examples = predictions_sorted_transposed[:, :m]
    negative_examples = predictions_sorted_transposed[:, m:]
    k = predictions_sorted_transposed.shape[0]

    tx = np.empty((k, m), dtype=float)
    ty = np.empty((k, n), dtype=float)
    tz = np.empty((k, m + n), dtype=float)

    for r in range(k):
        tx[r, :] = compute_midrank(positive_examples[r, :])
        ty[r, :] = compute_midrank(negative_examples[r, :])
        tz[r, :] = compute_midrank(predictions_sorted_transposed[r, :])

    aucs = tz[:, :m].sum(axis=1) / (m * n) - float(m + 1.0) / (2.0 * n)
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m

    sx = np.cov(v01)
    sy = np.cov(v10)
    delong_cov = sx / m + sy / n
    return aucs, delong_cov

In [13]:
# OOF 예측값 딕셔너리(oof_preds)를 사용한 DeLong test 함수
def compare_models_with_delong(oof_preds, y_true, model_a_name, model_b_name):
    preds_a = np.asarray(oof_preds[model_a_name])
    preds_b = np.asarray(oof_preds[model_b_name])
    y_true = np.asarray(y_true)

    assert np.array_equal(np.unique(y_true), [0, 1]), "y_true는 0/1 이진 레이블이어야 합니다."

    order = (-y_true).argsort()
    label_1_count = int(y_true.sum())

    preds_stacked = np.vstack((preds_a, preds_b))[:, order]
    aucs, delong_cov = fast_delong(preds_stacked, label_1_count)

    l = np.array([[1, -1]])
    var_diff = np.dot(np.dot(l, delong_cov), l.T)
    z = np.abs(aucs[0] - aucs[1]) / np.sqrt(var_diff)
    p_value = (2 * scipy.stats.norm.sf(z)).flatten()[0]

    print(f"\n[DeLong Test: {model_a_name} vs {model_b_name}]")
    print(f" - {model_a_name} AUC: {aucs[0]:.6f}")
    print(f" - {model_b_name} AUC: {aucs[1]:.6f}")
    print(f" - ΔAUC (B - A): {aucs[1] - aucs[0]:.6f}")
    print(f" - p-value: {p_value:.10f}")

    if p_value < 0.05:
        print(" >>> 결과: 두 모델의 성능 차이는 통계적으로 유의미합니다. (p < 0.05)")
    else:
        print(" >>> 결과: 두 모델의 성능 차이는 우연일 가능성이 있습니다. (p >= 0.05)")

    return {
        "model_a": model_a_name,
        "model_b": model_b_name,
        "auc_a": aucs[0],
        "auc_b": aucs[1],
        "delta_auc": aucs[1] - aucs[0],
        "p_value": p_value
    }

In [16]:
# 상위 2개 모델 비교
best_model = results_df.loc[0, "model"]
second_model = results_df.loc[1, "model"]
delong_result = compare_models_with_delong(oof_preds, y, best_model, second_model)


[DeLong Test: CatBoost vs LightGBM]
 - CatBoost AUC: 0.701791
 - LightGBM AUC: 0.693397
 - ΔAUC (B - A): -0.008394
 - p-value: 0.0011360678
 >>> 결과: 두 모델의 성능 차이는 통계적으로 유의미합니다. (p < 0.05)


- 최적 모델로 CatBoost 선택
- best model 기준 feature selection 진행 예정